<a href="https://colab.research.google.com/github/mixxr/gcolab-ml/blob/main/funnel/07_protein_structures_docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Episode 7 — Protein Structures & Docking
**Drug Discovery in Code** · [Life Sciences AI Applications]
Everything so far treated molecules as 2-D graphs. But a drug works by physically **fitting into a
pocket** on its target protein. **Docking** predicts *how* a molecule sits in that pocket and *how
well* it fits — a binding score. Today we download the real **EGFR** kinase structure, dock
**gefitinib** into its ATP site with **smina** (AutoDock Vina), and check the pose against the
crystal structure.
> Educational only — not medical advice.
> **Tooling:** this episode needs `smina` + `openbabel` (see `environment.yml`).
> `conda install -c conda-forge smina openbabel` (or the micromamba line in that file).

## 1. Get the target structure
PDB **1M17** is the EGFR kinase domain crystallised with **erlotinib** bound — a real drug in its
real pocket. We download it straight from the RCSB.

In [ ]:
import urllib.request, subprocess, re

urllib.request.urlretrieve(
    "https://files.rcsb.org/download/1M17.pdb", "../data/1M17.pdb")
print("downloaded 1M17.pdb")

## 2. Split receptor from the bound ligand
The protein atoms (`ATOM`, chain A) become the **receptor**; the co-crystallised drug
(`HETATM`, residue `AQ4` = erlotinib) becomes our **reference ligand** — we'll use it both to
place the search box and to judge our docked pose.

In [ ]:
lines = open("../data/1M17.pdb").read().splitlines()

receptor = [l for l in lines if l.startswith("ATOM") and l[21] == "A"]
reflig   = [l for l in lines if l.startswith("HETATM") and l[17:20] == "AQ4"]

open("../data/receptor.pdb", "w").write("\n".join(receptor))
open("../data/refligand.pdb", "w").write("\n".join(reflig))
print(len(receptor), "protein atoms |", len(reflig), "ligand atoms")

## 3. Prepare inputs (add H, charges → PDBQT)
Docking needs **PDBQT** files: hydrogens added, charges assigned. Open Babel does both. The
receptor is rigid (`-xr`); the query ligand **gefitinib** we build in 3-D straight from SMILES.

In [ ]:
def sh(cmd):
    print('$', cmd)
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout[-200:])

sh('obabel ../data/receptor.pdb  -O ../data/receptor.pdbqt  -xr -p 7.4')
sh('obabel ../data/refligand.pdb -O ../data/refligand.pdbqt')

gefitinib = 'COc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1OCCCN1CCOCC1'
sh(f'obabel -:"{gefitinib}" --gen3d -O ../data/gefitinib.pdbqt -p 7.4')

## 4. Dock
`smina` searches for the best pose. `--autobox_ligand` sizes the search box automatically around
the reference ligand — so we're docking into the *real* ATP site, not guessing coordinates.

In [ ]:
out = subprocess.run(
    "smina -r ../data/receptor.pdbqt -l ../data/gefitinib.pdbqt "
    "--autobox_ligand ../data/refligand.pdb --autobox_add 4 "
    "--exhaustiveness 8 --seed 42 --num_modes 9 "
    "-o ../data/gefitinib_docked.pdbqt",
    shell=True, capture_output=True, text=True)

# print the affinity table smina writes to stdout
print("\n".join(out.stdout.splitlines()[-13:]))

## 5. Read the score
Every pose in the output carries a `minimizedAffinity` in **kcal/mol** — more negative = tighter
predicted binding. Mode 1 is the best.

In [ ]:
poses = open("../data/gefitinib_docked.pdbqt").read()
scores = [float(x) for x in re.findall(r"minimizedAffinity (\S+)", poses)]
print("best affinity:", scores[0], "kcal/mol")
print("all modes:", scores)

## 6. Does the pose make sense?
The real test of a docking run isn't just the number — it's whether the pose is **physically
plausible**. Rendered in PyMOL, our docked gefitinib (yellow) lands right on top of the crystal
erlotinib (blue): the same quinazoline core threads into the ATP pocket, the halogenated aniline
points into the back cleft, and the solubilising tail exits toward solvent. Docking **recovered
the known binding mode** — that's what gives us confidence in the score.
```python
# rendering (needs pymol-open-source):
#   obabel gefitinib_docked.pdbqt -O pose.pdb -l 1   # best pose only
#   pymol -cq render.pml                             # cartoon + sticks -> png
```
![docked pose](../images/pose_pocket.png)

A binding score **and** a plausible pose, from a public structure and two open-source tools.

## Recap
- A drug acts by **fitting a pocket** — docking predicts the pose and a binding score.
- Real workflow: **fetch PDB → split receptor/ligand → prep PDBQT → smina → read affinity**.
- `--autobox_ligand` targets the true binding site using the co-crystal ligand.
- **Validate by re-docking**: our gefitinib pose reproduced the crystal binding mode.

**Next — Episode 8:** put it together at scale — **virtual screening**: score a whole library and
rank the best candidates.